#### Baseline OHLC Trading Strategy

In [2]:
from __future__ import annotations
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
import pandas as pd

REQUIRED_COLUMNS = ["Date", "Open", "High", "Low", "Close"]

@dataclass(frozen=True)
class StrategyParameters: # Parameters for baseline OHLC trading strategy
    initial_capital: float = 100_000.0

    # Previous day's open-to-close return boundaries
    previous_return_minimum: float = -0.10
    previous_return_maximum: float = -0.02

    # Current day's opening-gap boundaries
    opening_gap_minimum: float = -0.10
    opening_gap_maximum: float = 0.00

    # Trading costs applied on each side of the trade
    transaction_cost_bps_per_side: float = 2.5
    slippage_bps_per_side: float = 2.5

    trading_days_per_year: int = 252


# Load and validate daily OHLC data
def load_and_validate_ohlc(file_path: str | Path,) -> pd.DataFrame:

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"Dataset was not found: {file_path}")

    data = pd.read_csv(file_path)

    missing_columns = [column
        for column in REQUIRED_COLUMNS
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    data = data[REQUIRED_COLUMNS].copy()
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce",)

    for column in ["Open", "High", "Low", "Close"]:
        data[column] = pd.to_numeric(data[column], errors="coerce",)

    data = data.dropna(subset=REQUIRED_COLUMNS)
    data = data.sort_values("Date")
    data = data.drop_duplicates(subset="Date", keep="last",)

    valid_rows = (
        (data["Open"] > 0)
        & (data["High"] > 0)
        & (data["Low"] > 0)
        & (data["Close"] > 0)
        & (data["High"] >= data["Low"])
        & (data["High"] >= data["Open"])
        & (data["High"] >= data["Close"])
        & (data["Low"] <= data["Open"])
        & (data["Low"] <= data["Close"])
    )

    invalid_count = int((~valid_rows).sum())

    if invalid_count > 0:
        print(f"Removed {invalid_count} invalid OHLC rows.")

    data = data.loc[valid_rows].copy()
    data = data.set_index("Date")

    if len(data) < 2:
        raise ValueError("At least two valid trading days are required.")

    return data

# Generate daily trading signals: previous day's open-to-close return and current open relative to the previous close
def create_strategy_signals(ohlc: pd.DataFrame, parameters: StrategyParameters) -> pd.DataFrame:

    results = ohlc.copy()

    results["previous_open"] = results["Open"].shift(1)
    results["previous_close"] = results["Close"].shift(1)
    results["previous_day_return"] = results["previous_close"] / results["previous_open"] - 1.0
    results["opening_gap"] = results["Open"] / results["previous_close"] - 1.0

    previous_decline_condition = ((results["previous_day_return"] >= parameters.previous_return_minimum)
        & (results["previous_day_return"] < parameters.previous_return_maximum))

    gap_down_condition = ((results["opening_gap"] >= parameters.opening_gap_minimum)
        & (results["opening_gap"] < parameters.opening_gap_maximum))

    results["previous_decline_condition"] = previous_decline_condition
    results["gap_down_condition"] = gap_down_condition
    results["entry_signal"] = previous_decline_condition & gap_down_condition
    results["signal"] = np.where(results["entry_signal"], "BUY_OPEN_SELL_CLOSE", "NO_TRADE",)

    return results.dropna(subset=[
            "previous_open",
            "previous_close",
            "previous_day_return",
            "opening_gap",
        ]
    )


# Execute daily open-to-close strategy: buy at the current open and sell at the current close
def run_daily_strategy(ohlc: pd.DataFrame, parameters: StrategyParameters) -> pd.DataFrame:

    results = create_strategy_signals(ohlc=ohlc,parameters=parameters)
    transaction_cost_rate = parameters.transaction_cost_bps_per_side / 10_000.0
    slippage_rate = parameters.slippage_bps_per_side / 10_000.0
    results["position"] = results["entry_signal"].astype(int)

    # Actual execution prices after slippage
    results["entry_price"] = np.where(
        results["position"] == 1,
        results["Open"] * (1.0 + slippage_rate),
        np.nan,
    )

    results["exit_price"] = np.where(
        results["position"] == 1,
        results["Close"] * (1.0 - slippage_rate),
        np.nan,
    )

    # Raw open-to-close price return before costs
    results["gross_open_to_close_return"] = (
        results["Close"] / results["Open"] - 1.0
    )

    # Return based on slippage-adjusted execution prices
    results["return_after_slippage"] = np.where(
        results["position"] == 1,
        results["exit_price"]
        / results["entry_price"]
        - 1.0,
        0.0,
    )

    # Commission or transaction costs on entry and exit
    results["transaction_cost_return"] = np.where(
        results["position"] == 1,
        2.0 * transaction_cost_rate,
        0.0,
    )

    results["net_strategy_return"] = (
        results["return_after_slippage"]
        - results["transaction_cost_return"]
    )

    # No-trade days must have a zero return
    results.loc[
        results["position"] == 0,
        "net_strategy_return",
    ] = 0.0

    results["gross_strategy_return"] = np.where(
        results["position"] == 1,
        results["gross_open_to_close_return"],
        0.0,
    )

    results["portfolio_value"] = (
        parameters.initial_capital
        * (
            1.0 + results["net_strategy_return"]
        ).cumprod()
    )

    results["cumulative_return"] = (
        results["portfolio_value"]
        / parameters.initial_capital
        - 1.0
    )

    results["running_peak"] = (
        results["portfolio_value"].cummax()
    )

    results["drawdown"] = (
        results["portfolio_value"]
        / results["running_peak"]
        - 1.0
    )

    results["trade_outcome"] = np.select(
        [
            (
                (results["position"] == 1)
                & (results["net_strategy_return"] > 0)
            ),
            (
                (results["position"] == 1)
                & (results["net_strategy_return"] < 0)
            ),
            (
                (results["position"] == 1)
                & (results["net_strategy_return"] == 0)
            ),
        ],
        [
            "WIN",
            "LOSS",
            "BREAKEVEN",
        ],
        default="NO_TRADE",
    )

    return results


# Create one trade record for each qualifying trading day
def create_trade_log(daily_results: pd.DataFrame,parameters: StrategyParameters) -> pd.DataFrame:
   
    trades = daily_results.loc[daily_results["position"] == 1].copy()

    if trades.empty:
        return pd.DataFrame(
            columns=[
                "trade_id",
                "trade_date",
                "action_open",
                "action_close",
                "entry_price",
                "exit_price",
                "previous_day_return",
                "opening_gap",
                "gross_trade_return",
                "net_trade_return",
                "trade_outcome",
                "portfolio_value",
            ]
        )

    trades = trades.reset_index()
    trades.insert(0, "trade_id", np.arange(1, len(trades) + 1))

    trades["action_open"] = "BUY_OPEN"
    trades["action_close"] = "SELL_CLOSE"

    trades = trades.rename(
        columns={
            "Date": "trade_date",
            "gross_open_to_close_return": "gross_trade_return",
            "net_strategy_return": "net_trade_return",
        }
    )

    trade_columns = [
        "trade_id",
        "trade_date",
        "action_open",
        "action_close",
        "Open",
        "Close",
        "entry_price",
        "exit_price",
        "previous_open",
        "previous_close",
        "previous_day_return",
        "opening_gap",
        "gross_trade_return",
        "return_after_slippage",
        "transaction_cost_return",
        "net_trade_return",
        "trade_outcome",
        "portfolio_value",
        "cumulative_return",
    ]

    return trades[trade_columns].copy()


# Calculate portfolio and trade performance metrics
def calculate_performance_metrics(daily_results: pd.DataFrame,parameters: StrategyParameters) -> dict[str, float]:

    if daily_results.empty:
        raise ValueError("The daily results DataFrame is empty.")

    daily_returns = (daily_results["net_strategy_return"].astype(float).fillna(0.0))
    trades = daily_results.loc[daily_results["position"] == 1].copy()
    initial_capital = parameters.initial_capital
    final_portfolio_value = float(daily_results["portfolio_value"].iloc[-1])
    net_profit = final_portfolio_value - initial_capital
    total_return = final_portfolio_value / initial_capital - 1.0
    number_of_days = len(daily_returns)
    number_of_years = number_of_days / parameters.trading_days_per_year
    
    annualized_return = (
        (final_portfolio_value / initial_capital)
        ** (1.0 / number_of_years) - 1.0
        if (number_of_years > 0 and final_portfolio_value > 0)
        else np.nan
    )
    annualized_volatility = (daily_returns.std(ddof=1) * np.sqrt(parameters.trading_days_per_year))
    return_standard_deviation = daily_returns.std(ddof=1)
    
    sharpe_ratio = (
        daily_returns.mean() / return_standard_deviation * np.sqrt(parameters.trading_days_per_year)
        if return_standard_deviation > 0
        else np.nan
    )
    downside_returns = daily_returns.clip(upper=0.0)
    daily_downside_deviation = np.sqrt(np.mean(np.square(downside_returns)))
    annualized_downside_deviation = (daily_downside_deviation * np.sqrt(parameters.trading_days_per_year))
    
    sortino_ratio = (
        daily_returns.mean()
        * parameters.trading_days_per_year
        / annualized_downside_deviation
        if annualized_downside_deviation > 0
        else np.nan
    )
    maximum_drawdown = float(daily_results["drawdown"].min())

    calmar_ratio = (
        annualized_return
        / abs(maximum_drawdown)
        if maximum_drawdown < 0
        else np.nan
    )

    number_of_trades = len(trades)
    winning_trades = int((trades["net_strategy_return"] > 0).sum())
    losing_trades = int((trades["net_strategy_return"] < 0).sum())
    breakeven_trades = int((trades["net_strategy_return"] == 0).sum())

    trade_win_rate = (
        winning_trades / number_of_trades
        if number_of_trades > 0
        else np.nan
    )

    average_net_trade_return = (
        float(trades["net_strategy_return"].mean())
        if number_of_trades > 0
        else np.nan
    )

    median_net_trade_return = (
        float(trades["net_strategy_return"].median())
        if number_of_trades > 0
        else np.nan
    )

    gross_profit = float(trades.loc[trades["net_strategy_return"] > 0,"net_strategy_return",].sum())
    gross_loss = abs(float(trades.loc[trades["net_strategy_return"] < 0,"net_strategy_return",].sum()))

    profit_factor = (
        gross_profit / gross_loss
        if gross_loss > 0
        else np.inf
        if gross_profit > 0
        else np.nan
    )

    trade_frequency = (
        number_of_trades / number_of_days
        if number_of_days > 0
        else np.nan
    )

    return {
        "Initial Capital": initial_capital,
        "Final Portfolio Value": final_portfolio_value,
        "Net Profit": net_profit,
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_volatility,
        "Sharpe Ratio": sharpe_ratio,
        "Sortino Ratio": sortino_ratio,
        "Maximum Drawdown": maximum_drawdown,
        "Calmar Ratio": calmar_ratio,
        "Number of Trading Days": float(number_of_days),
        "Number of Trades": float(number_of_trades),
        "Winning Trades": float(winning_trades),
        "Losing Trades": float(losing_trades),
        "Breakeven Trades": float(breakeven_trades),
        "Trade Win Rate": trade_win_rate,
        "Average Net Trade Return": average_net_trade_return,
        "Median Net Trade Return": median_net_trade_return,
        "Profit Factor": profit_factor,
        "Trade Frequency": trade_frequency,
    }


# Print performance metrics
def print_performance_metrics(metrics: dict[str, float]) -> None:

    percentage_metrics = {
        "Total Return",
        "Annualized Return",
        "Annualized Volatility",
        "Maximum Drawdown",
        "Trade Win Rate",
        "Average Net Trade Return",
        "Median Net Trade Return",
        "Trade Frequency",
    }

    currency_metrics = {
        "Initial Capital",
        "Final Portfolio Value",
        "Net Profit",
    }

    integer_metrics = {
        "Number of Trading Days",
        "Number of Trades",
        "Winning Trades",
        "Losing Trades",
        "Breakeven Trades",
    }

    print("=" * 65)
    print("Baseline OHLC Trading Strategy")
    print("=" * 65)

    for metric_name, metric_value in metrics.items():
        if pd.isna(metric_value):
            formatted_value = "N/A"
        elif metric_name in percentage_metrics:
            formatted_value = f"{metric_value:.2%}"
        elif metric_name in currency_metrics:
            formatted_value = f"${metric_value:,.2f}"
        elif metric_name in integer_metrics:
            formatted_value = f"{int(metric_value):,}"
        elif np.isinf(metric_value):
            formatted_value = "Infinity"
        else:
            formatted_value = f"{metric_value:.4f}"
        print(
            f"{metric_name:<34}"
            f"{formatted_value:>20}"
        )
    print("=" * 65)


# Perform basic consistency checks on the strategy results
def validate_strategy_results(daily_results: pd.DataFrame) -> None:
   
    valid_signals = {"BUY_OPEN_SELL_CLOSE","NO_TRADE",}
    observed_signals = set(daily_results["signal"].unique())
    invalid_signals = (observed_signals - valid_signals)

    if invalid_signals:
        raise ValueError(f"Invalid signals detected: {invalid_signals}")

    no_trade_returns = daily_results.loc[daily_results["position"] == 0,"net_strategy_return",]

    if not np.allclose(no_trade_returns,0.0,):
        raise ValueError("No-trade days contain non-zero strategy returns.")

    trade_rows = daily_results.loc[daily_results["position"] == 1]

    if trade_rows[["entry_price", "exit_price"]].isna().any().any():
        raise ValueError("One or more trades have missing execution prices.")

    if (daily_results["portfolio_value"] <= 0).any():
        raise ValueError("Portfolio value became zero or negative.")

    print("Strategy result validation passed.")


# Run the complete strategy, save all outputs and returns daily_results, trade_log, metrics_table
def run_complete_strategy(
    file_path: str | Path,
    output_directory: str | Path = ".",
    parameters: StrategyParameters | None = None,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    
    if parameters is None:
        parameters = StrategyParameters()

    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True,exist_ok=True,)

    ohlc = load_and_validate_ohlc(file_path)
    daily_results = run_daily_strategy(ohlc=ohlc,parameters=parameters,)
    trade_log = create_trade_log(daily_results=daily_results,parameters=parameters,)

    metrics = calculate_performance_metrics(daily_results=daily_results,parameters=parameters,)
    metrics_table = pd.DataFrame(
        {
            "Metric": list(metrics.keys()),
            "Value": list(metrics.values()),
        }
    )

    validate_strategy_results(daily_results)
    print_performance_metrics(metrics)

    daily_output_path = (output_directory / "daily_strategy_results.csv")
    trade_output_path = (output_directory / "daily_trade_records.csv")
    metrics_output_path = (output_directory / "daily_strategy_metrics.csv")
    parameters_output_path = (output_directory / "daily_strategy_parameters.csv")
    
    daily_results.to_csv(daily_output_path,index=True,)
    trade_log.to_csv(trade_output_path,index=False,)
    metrics_table.to_csv(metrics_output_path,index=False,)

    pd.DataFrame([asdict(parameters)]).to_csv(parameters_output_path,index=False,)

    print("\nFiles saved:")
    print(f"Daily results: {daily_output_path}")
    print(f"Trade records: {trade_output_path}")
    print(f"Metrics: {metrics_output_path}")
    print(f"Parameters: {parameters_output_path}")

    return (
        daily_results,
        trade_log,
        metrics_table,
    )

In [3]:
parameters = StrategyParameters(
    initial_capital=100_000.0,

    previous_return_minimum=-0.10,
    previous_return_maximum=-0.02,

    opening_gap_minimum=-0.10,
    opening_gap_maximum=0.00,

    transaction_cost_bps_per_side=2.5,
    slippage_bps_per_side=2.5,

    trading_days_per_year=252,
)

daily_results, trade_log, metrics_table = (
    run_complete_strategy(
        file_path="data/cleaned_market_data.csv",
        output_directory="data/model1",
        parameters=parameters,
    )
)

# Save another daily result as the baseline
daily_results.to_csv("data/baseline_strategy.csv")

strategy1_metrics = (metrics_table.set_index("Metric")["Value"].to_dict())

Strategy result validation passed.
Baseline OHLC Trading Strategy
Initial Capital                            $100,000.00
Final Portfolio Value                      $137,970.00
Net Profit                                  $37,970.00
Total Return                                    37.97%
Annualized Return                                5.54%
Annualized Volatility                            7.24%
Sharpe Ratio                                    0.7811
Sortino Ratio                                   1.3791
Maximum Drawdown                                -5.52%
Calmar Ratio                                    1.0045
Number of Trading Days                           1,504
Number of Trades                                    79
Winning Trades                                      47
Losing Trades                                       32
Breakeven Trades                                     0
Trade Win Rate                                  59.49%
Average Net Trade Return                         0.43%